# 01. BigEarthNet-S2: 12-Band Multispectral Data Exploration & Analysis

This notebook explores the **12-band Sentinel-2 Level-2A** multispectral benchmark archive for land-cover classification.

> [!IMPORTANT]
> **Multi-Label Formulation**: BigEarthNet-S2 is strictly a **multi-label classification** dataset.
> Each satellite patch ($1.2\text{ km} \times 1.2\text{ km}$) naturally contains heterogeneous land-cover mixtures.
> BigEarthNet provides patch-level multi-label land-cover annotations, **not** bounding boxes or pixel-level segmentation masks.

## Section 1: 12 Sentinel-2 Bands & Spatial Grid Alignment
Sentinel-2 Level-2A products provide 12 surface reflectance bands at three native resolutions (10m, 20m, 60m). We align all bands to the standard $120 \times 120$ spatial grid.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().parent.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ml.landcover.src.labels import (
    BIGEARTHNET_19_CLASSES,
    SENTINEL2_12_BANDS,
    BAND_NAMES_12,
    CLASS_TO_IDX,
    CLC43_TO_BEN19,
)

bands_df = pd.DataFrame(SENTINEL2_12_BANDS)
print(f'Configured Sentinel-2 Bands: {len(bands_df)}')
bands_df[['band', 'name', 'wavelength_nm', 'native_gsd_m', 'native_grid']]

### Spatial Grid Alignment Rationale
* 10m bands (B02, B03, B04, B08) have native $120 \times 120$ resolution.
* 20m bands (B05, B06, B07, B8A, B11, B12) are bilinearly upsampled by $2\times$ from $60 \times 60$ to $120 \times 120$.
* 60m bands (B01, B09) are bilinearly upsampled by $6\times$ from $20 \times 20$ to $120 \times 120$.
This constructs a uniform tensor $\mathbf{X} \in \mathbb{R}^{12 \times 120 \times 120}$ compatible with the SeCo 12-channel ResNet-50 architecture.

## Section 2: BigEarthNet-19 Label Hierarchy & Imbalance
We inspect representative 12-band Sentinel-2 patches and analyze the distribution of the 19 standard classes.

In [ ]:
from ml.landcover.src.dataset import create_synthetic_bigearthnet_samples
data_dir = repo_root / 'ml' / 'landcover' / 'data' / 'samples'
samples = create_synthetic_bigearthnet_samples(str(data_dir), num_samples=80)
print(f'Loaded {len(samples)} representative 12-band Sentinel-2 samples.')

class_counts = {c: 0 for c in BIGEARTHNET_19_CLASSES}
multiplicity = []
for s in samples:
    multiplicity.append(len(s['labels']))
    for lbl in s['labels']:
        if lbl in class_counts:
            class_counts[lbl] += 1

print(f'Mean labels per patch: {np.mean(multiplicity):.2f} (Range: {min(multiplicity)} - {max(multiplicity)})')

In [ ]:
plt.figure(figsize=(10, 6), dpi=120)
sorted_counts = sorted(class_counts.items(), key=lambda x: x[1])
names = [x[0] for x in sorted_counts]
counts = [x[1] for x in sorted_counts]
plt.barh(names, counts, color='#2980b9', edgecolor='black', height=0.6)
plt.xlabel('Frequency in Dataset Split', fontweight='bold')
plt.title('BigEarthNet-19 Class Distribution', fontsize=12, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Section 3: Multi-Label Co-occurrence Matrix
We compute pairwise co-occurrences between the 19 land-cover classes.

In [ ]:
co_matrix = np.zeros((len(BIGEARTHNET_19_CLASSES), len(BIGEARTHNET_19_CLASSES)), dtype=int)
for s in samples:
    indices = [CLASS_TO_IDX[lbl] for lbl in s['labels'] if lbl in CLASS_TO_IDX]
    for i in indices:
        for j in indices:
            co_matrix[i, j] += 1

plt.figure(figsize=(10, 8), dpi=120)
plt.imshow(co_matrix, cmap='YlGnBu', interpolation='nearest')
plt.colorbar(label='Co-occurrence Count')
plt.xticks(range(len(BIGEARTHNET_19_CLASSES)), BIGEARTHNET_19_CLASSES, rotation=90, fontsize=8)
plt.yticks(range(len(BIGEARTHNET_19_CLASSES)), BIGEARTHNET_19_CLASSES, fontsize=8)
plt.title('Multi-Label Land-Cover Co-occurrence Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 4: Multispectral Spectral Profiles Across 12 Bands
We inspect the reflectance signatures across the 12 Sentinel-2 bands for different land-cover archetypes.

In [ ]:
from ml.landcover.src.dataset import load_12band_sentinel2_raster, extract_rgb_composite
sample_cube = load_12band_sentinel2_raster(samples[0]['image_path'])
rgb_img = extract_rgb_composite(sample_cube)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), dpi=120)
axes[0].imshow(rgb_img)
axes[0].set_title(f"True-Color RGB (B04-B03-B02)\nLabels: {', '.join(samples[0]['labels'])}", fontsize=10)
axes[0].axis('off')

band_means = sample_cube.mean(axis=(1, 2))
axes[1].plot(BAND_NAMES_12, band_means, 'o-', color='#e67e22', linewidth=2, markersize=6)
axes[1].set_xlabel('Sentinel-2 Band', fontweight='bold')
axes[1].set_ylabel('Mean BOA Reflectance', fontweight='bold')
axes[1].set_title('12-Band Spectral Signature', fontsize=11, fontweight='bold')
axes[1].grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

### Summary of Data Exploration

### Q&A
* **Why use all 12 Sentinel-2 bands?** The 12-band configuration ensures full architectural compatibility with the SeCo-ECO ResNet-50 checkpoint and leverages vegetation red-edge and SWIR bands.
* **Does BigEarthNet provide bounding boxes or segmentation masks?** No, it provides patch-level multi-label land-cover annotations.

### Data Analysis Key Findings
* 12 bands are aligned to the common $120 \times 120$ spatial grid using bilinear resampling for 20m and 60m bands.
* Multi-label co-occurrence confirms frequent co-existence between agricultural and forest categories.
* Multi-band reflectances exhibit distinct spectral signatures distinguishing vegetation, water, and built environments.

### Insights or Next Steps
* Proceed to `02_baseline.ipynb` to establish Tier 1 (Class Frequency) and Tier 2 (Frozen SeCo linear probe) baselines.